# 00 — Data Acquisition

Fetches the **raw** external inputs and caches them. Re-running is cheap: an
existing cache is loaded and the download is skipped (set `force_refresh: true`
in `config.yaml`, or `force_refresh=True` per call, to re-download).

**Raw only.** This stage does *not* compute returns, volatility, technical
indicators, `days_since_fomc`, or any lag. Those look-ahead-sensitive transforms
live in the feature layer (`01_features_target`) so every lag sits in one place.

Outputs (in `cache/`):
- `indices_raw.parquet` — OHLCV price panel, columns `(Field, Ticker)`
- `macro_raw.parquet` — FRED series (daily, forward-filled) + `*_pct_change`
- `fomc_calendar_2000_present.csv` — FOMC meeting/decision dates

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import load_config
from src.data import (
    load_or_fetch,
    download_price_panel,
    apply_vvix_gate,
    download_fred,
    scrape_fomc_calendar,
)

# Existing root config modules (ticker lists)
from index_ticker import SP500, US_INDEX_TICKERS, MACRO_TICKERS_POST_2006, INT_TICKERS
from fred_macro import FRED_MACRO_TICKERS

cfg = load_config(PROJECT_ROOT / "config.yaml")
print(f"window     : {cfg.start} -> {cfg.end}  (ref from {cfg.ref_start})")
print(f"cache dir  : {cfg.cache_dir}")
print(f"force_refresh: {cfg.force_refresh}")

## 1 — Price panel (yfinance)

`primary` (target + US indices) share the NYSE calendar. `reference` (VIX/VVIX,
DXY, international indices) may trade on other calendars, so they are forward-filled
onto the primary calendar. VVIX is dropped at load time when `start` predates VVIX
history (`vvix_min_start`), keeping the cache itself universal.

In [ ]:
primary = [SP500] + US_INDEX_TICKERS
reference = MACRO_TICKERS_POST_2006 + INT_TICKERS   # VIX, VVIX, DXY + international

panel = load_or_fetch(
    cfg.indices_path,
    lambda: download_price_panel(
        primary=primary,
        reference=reference,
        start=cfg.start,
        end=cfg.end,
        ref_start=cfg.ref_start,
    ),
    force_refresh=cfg.force_refresh,
)

# Deterministic load-time filter (see docstring): drop VVIX if start < 2007.
panel = apply_vvix_gate(panel, cfg.start, vvix_min_start=cfg.vvix_min_start)

panel.tail(3)

## 2 — Macro (FRED)

Set your FRED key in the environment variable named by `acquire.fred_api_key_env`
(default `FRED_API_KEY`), e.g. `export FRED_API_KEY=...`. Series come from the
existing `fred_macro.FRED_MACRO_TICKERS`. Daily-series lagging happens later, in the
feature layer.

In [ ]:
api_key = cfg.fred_api_key
if not api_key:
    raise RuntimeError(
        f"FRED API key not found. Set the {cfg.fred_api_key_env!r} environment "
        f"variable before running this cell."
    )

macro = load_or_fetch(
    cfg.macro_path,
    lambda: download_fred(
        FRED_MACRO_TICKERS,
        start=cfg.ref_start,   # a little history before `start` for forward-fill
        end=cfg.end,
        api_key=api_key,
        pause=cfg.request_pause_sec,
    ),
    force_refresh=cfg.force_refresh,
)

macro.tail(3)

## 3 — FOMC calendar

Historical archive pages (2000–2020) plus the live calendar page (2021+). Cached as
CSV for easy inspection. `days_since_fomc` is derived later against the trading
calendar, in the feature layer.

In [ ]:
fomc = load_or_fetch(
    cfg.fomc_path,
    lambda: scrape_fomc_calendar(2000, 2020, include_recent=True, pause=cfg.request_pause_sec),
    force_refresh=cfg.force_refresh,
    saver=lambda df, p: df.to_csv(p, index=False),
    loader=lambda p: pd.read_csv(p, parse_dates=["date"]),
)

fomc.tail(3)

## 4 — Sanity summary

In [ ]:
tickers = sorted(set(panel.columns.get_level_values("Ticker")))

print("PANEL")
print(f"  shape   : {panel.shape}")
print(f"  dates   : {panel.index.min().date()} -> {panel.index.max().date()}")
print(f"  tickers : {tickers}")
print(f"  VVIX in : {'^VVIX' in tickers}  (start {cfg.start} vs min {cfg.vvix_min_start})")

print("\nMACRO")
print(f"  shape   : {macro.shape}")
print(f"  dates   : {macro.index.min().date()} -> {macro.index.max().date()}")
levels = [c for c in macro.columns if not str(c).endswith('_pct_change')]
print(f"  series  : {levels}")

print("\nFOMC")
print(f"  shape        : {fomc.shape}")
print(f"  dates        : {fomc['date'].min().date()} -> {fomc['date'].max().date()}")
print(f"  decision days: {int(fomc['is_fomc_day'].sum())}")

# Missingness on the modelled window (post-warmup NaNs worth eyeballing):
window = panel.loc[pd.Timestamp(cfg.start):]
na_frac = window.isna().mean().sort_values(ascending=False)
print("\nTop NaN fractions in panel (from start date):")
print(na_frac.head(8).to_string())